In [1]:
!pip install -U langchain langchain-core langchain-community langchain-openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.0/473.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully 

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()

# 랭체인 기본 : LLM 연결 (추상화)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)
response = llm.invoke("랭체인이 뭐냐?")
print(response.content)

랭체인(LangChain)은 대형 언어 모델(LLM, Large Language Model)을 활용한 애플리케이션 개발을 쉽게 해주는 오픈소스 프레임워크입니다. 

기본적으로 랭체인은 LLM과 외부 데이터, 도구, API 등을 연결하여 복잡한 작업을 수행할 수 있도록 도와줍니다. 예를 들어, 단순한 질의응답 뿐만 아니라 문서 요약, 대화형 에이전트, 데이터 검색, 자동화된 워크플로우 등을 만들 때 사용됩니다.

랭체인의 주요 특징은 다음과 같습니다:

1. **체인(Chain)**: 여러 개의 작업(step)을 순차적으로 연결하여 복잡한 프로세스를 구성할 수 있습니다.
2. **에이전트(Agent)**: LLM이 스스로 외부 도구나 API를 호출하며 문제를 해결하도록 하는 기능입니다.
3. **메모리(Memory)**: 대화나 작업 과정에서 상태를 기억하여 더 자연스럽고 연속적인 상호작용을 가능하게 합니다.
4. **통합(Integration)**: 벡터 데이터베이스, 문서 저장소, 검색 엔진 등 다양한 외부 시스템과 쉽게 연결할 수 있습니다.

즉, 랭체인은 LLM을 단순한 텍스트 생성기가 아니라, 실제 애플리케이션에서 유용하게 활용할 수 있도록 구조화하고 확장하는 데 도움을 주는 툴킷이라고 볼 수 있습니다.


In [10]:
# PromptTemplate 사용 : 기본 프롬프트 + 변수 삽입
# 페르소나
mytemplate = """너는 한국어 전문가야.
아래 내용으로 5행 이내의 아름다운 시를 작성해 줘.
'{content}'
"""

print("template:", mytemplate)

prompt = PromptTemplate(
    input_variables=["content"],
    template=mytemplate,
)

fill_prompt = prompt.format(content="가을하늘")
print("fill_prompt:", fill_prompt)

poem_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.9)

poem_response = poem_llm.invoke(fill_prompt)
print(poem_response.content)

template: 너는 한국어 전문가야.
아래 내용으로 5행 이내의 아름다운 시를 작성해 줘.
'{content}'

fill_prompt: 너는 한국어 전문가야.
아래 내용으로 5행 이내의 아름다운 시를 작성해 줘.
'가을하늘'

가을하늘 푸르름 가득 안고  
바람은 살며시 속삭이네  
낙엽은 춤추듯 내려와  
마음 깊이 스며드는 그 빛  
추억 속에 물드는 가을밤


In [17]:
# tool 사용 (tool + Agent 구조)
print('계산기 툴 작성 ----')
from langchain.tools import tool
from langchain.agents import create_agent

# Tool 정의
@tool
def myCulc(expression: str) -> str:
  # 간단한 설명 필요 description
  """간단한 사칙연산 수식을 계산하고 '수식=값' 형태의 문자열로 반환한다."""
  try:
    result = eval(expression)    # 문자열 수식(expression)을 연산해서 숫자로 반환
    return f"{expression} = {result}"
  except Exception as e:
    return f"계산 실패 : {e}"

tools = [myCulc]

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "너는 수학 계산을 도와주는 어시스턴트야"
        "가능하면 myCulc 툴을 사용해 정확한 값을 계산해줘"
        "출력은 형식을 지켜서 답해. 수식=값"
    )
)

question = "6 * (3+2) / 2는 얼마야?"
result = agent.invoke({
    "message":[
        {"role":"user", "content":question}
    ]
})

# print(result)
last_msg = result["messages"][-1]
print("최종 답변 : ", last_msg.content)

계산기 툴 작성 ----
최종 답변 :  2+2=4
3*5=15
